# Muestreo, Ventanas y Descomposición Espectral

Este notebook explora el análisis espectral de señales mediante el cálculo del periodograma, el efecto del muestreo y la aplicación de diferentes ventanas.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import signal
from scipy.fft import fft
from scipy.io import wavfile
import librosa

## 1. Periodograma mediante FFT

El periodograma es un estimador de la densidad de potencia espectral (PSD) basado en la transformada de Fourier.

In [ ]:
# Generar señal de ejemplo
fs = 1000  # Hz
t = np.arange(0, 1, 1/fs)
x = np.cos(2*np.pi*100*t) + np.random.randn(len(t))

# Graficar señal en el dominio del tiempo
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=x, mode='lines', name='x(t)'))
fig.update_layout(
    title='Señal en el dominio del tiempo',
    xaxis_title='Tiempo (s)',
    yaxis_title='Amplitud',
    template='plotly_white'
)
fig.show()

In [ ]:
# Cálculo de FFT
N = len(x)
xdft = fft(x)

# Magnitud de la transformada de Fourier
fig = go.Figure()
fig.add_trace(go.Scatter(y=np.abs(xdft), mode='lines'))
fig.update_layout(
    title='Magnitud de la FFT',
    xaxis_title='Índice de frecuencia',
    yaxis_title='Magnitud',
    template='plotly_white'
)
fig.show()

In [ ]:
# Cálculo del periodograma
xdft = xdft[:N//2+1]
psdx = (1/(fs*N)) * np.abs(xdft)**2
psdx[1:-1] = 2*psdx[1:-1]
freq = np.linspace(0, fs/2, len(xdft))

# Graficar periodograma
fig = go.Figure()
fig.add_trace(go.Scatter(x=freq, y=10*np.log10(psdx), mode='lines'))
fig.update_layout(
    title='Periodograma usando FFT',
    xaxis_title='Frecuencia (Hz)',
    yaxis_title='PSD (dB/Hz)',
    template='plotly_white'
)
fig.show()

In [ ]:
# Periodograma usando scipy
f, Pxx = signal.periodogram(x, fs, window='boxcar', scaling='density')

fig = go.Figure()
fig.add_trace(go.Scatter(x=f, y=Pxx, mode='lines'))
fig.update_layout(
    title='Periodograma usando scipy.signal',
    xaxis_title='Frecuencia (Hz)',
    yaxis_title='PSD (V²/Hz)',
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

## 2. Efecto del Muestreo

Análisis del efecto de la frecuencia de muestreo sobre el espectro de la señal.

In [ ]:
# Definir señal continua
def x_continua(t):
    return np.cos(4*2*np.pi*t) + np.cos(4.5*2*np.pi*t)

# Graficar aproximación de señal continua
t_continuo = np.linspace(0, 10, 10000)
x_cont = x_continua(t_continuo)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_continuo, y=x_cont, mode='lines'))
fig.update_layout(
    title='Señal en tiempo continuo',
    xaxis_title='Tiempo (s)',
    yaxis_title='Amplitud',
    xaxis_range=[0, 5],
    template='plotly_white'
)
fig.show()

In [ ]:
# Muestreo con fs = 3 Hz (submuestreo)
fs1 = 3  # Hz
t1 = np.arange(0, 40, 1/fs1)
x1 = x_continua(t1)
N1 = len(x1)

f1, Pxx1 = signal.periodogram(x1, fs1, window='boxcar', scaling='density')

fig = go.Figure()
fig.add_trace(go.Scatter(x=f1, y=Pxx1, mode='lines'))
fig.update_layout(
    title=f'Periodograma con fs = {fs1} Hz',
    xaxis_title='Frecuencia (Hz)',
    yaxis_title='PSD (V²/Hz)',
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

print(f'Número de muestras: {N1}')

In [ ]:
# Muestreo con fs = 12 Hz
fs2 = 12  # Hz
t2 = np.arange(0, 40, 1/fs2)
x2 = x_continua(t2)
N2 = len(x2)

f2, Pxx2 = signal.periodogram(x2, fs2, window='boxcar', scaling='density')

fig = go.Figure()
fig.add_trace(go.Scatter(x=f2, y=Pxx2, mode='lines'))
fig.update_layout(
    title=f'Periodograma con fs = {fs2} Hz',
    xaxis_title='Frecuencia (Hz)',
    yaxis_title='PSD (V²/Hz)',
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

print(f'Número de muestras: {N2}')

## 3. Aplicación de Ventanas

Las ventanas modifican las características espectrales de la señal, permitiendo mejorar la resolución en frecuencia o el rango dinámico.

In [ ]:
# Señal con componentes muy cercanas en frecuencia
def x_ventanas(t):
    return np.cos(4*2*np.pi*t) + 0.001*np.cos(4.2*2*np.pi*t)

fs3 = 100  # Hz
t3 = np.arange(0, 40, 1/fs3)
x3 = x_ventanas(t3)
N3 = len(x3)

print(f'Número de muestras: {N3}')

In [ ]:
# Ventana rectangular (por defecto)
f_rect, Pxx_rect = signal.periodogram(x3, fs3, window='boxcar', scaling='density')

fig = go.Figure()
fig.add_trace(go.Scatter(x=f_rect, y=Pxx_rect, mode='lines'))
fig.update_layout(
    title='Periodograma con ventana rectangular',
    xaxis_title='Frecuencia (Hz)',
    yaxis_title='PSD (V²/Hz)',
    xaxis_range=[0, 10],
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

In [ ]:
# Ventana flat-top (alto rango dinámico)
f_flat, Pxx_flat = signal.periodogram(x3, fs3, window='flattop', scaling='density')

fig = go.Figure()
fig.add_trace(go.Scatter(x=f_flat, y=Pxx_flat, mode='lines'))
fig.update_layout(
    title='Periodograma con ventana flat-top (alto rango dinámico)',
    xaxis_title='Frecuencia (Hz)',
    yaxis_title='PSD (V²/Hz)',
    xaxis_range=[0, 10],
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

In [ ]:
# Ventana Blackman-Harris (alta resolución)
f_bh, Pxx_bh = signal.periodogram(x3, fs3, window='blackmanharris', scaling='density')

fig = go.Figure()
fig.add_trace(go.Scatter(x=f_bh, y=Pxx_bh, mode='lines'))
fig.update_layout(
    title='Periodograma con ventana Blackman-Harris (alta resolución)',
    xaxis_title='Frecuencia (Hz)',
    yaxis_title='PSD (V²/Hz)',
    xaxis_range=[0, 10],
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

In [ ]:
# Comparación de ventanas
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Ventana rectangular', 'Ventana flat-top', 'Ventana Blackman-Harris'),
    vertical_spacing=0.1
)

fig.add_trace(go.Scatter(x=f_rect, y=Pxx_rect, mode='lines', name='Rectangular'), row=1, col=1)
fig.add_trace(go.Scatter(x=f_flat, y=Pxx_flat, mode='lines', name='Flat-top'), row=2, col=1)
fig.add_trace(go.Scatter(x=f_bh, y=Pxx_bh, mode='lines', name='Blackman-Harris'), row=3, col=1)

fig.update_xaxes(title_text='Frecuencia (Hz)', range=[0, 10], row=3, col=1)
fig.update_xaxes(range=[0, 10], row=1, col=1)
fig.update_xaxes(range=[0, 10], row=2, col=1)

fig.update_yaxes(title_text='PSD (V²/Hz)', type='log', row=1, col=1)
fig.update_yaxes(title_text='PSD (V²/Hz)', type='log', row=2, col=1)
fig.update_yaxes(title_text='PSD (V²/Hz)', type='log', row=3, col=1)

fig.update_layout(
    height=900,
    showlegend=False,
    template='plotly_white'
)
fig.show()

## 4. Análisis de Archivos de Audio

In [ ]:
# Cargar archivo WAV
fs_wav, audio_wav = wavfile.read('../../data/Owl.wav')

# Convertir a mono si es estéreo
if len(audio_wav.shape) > 1:
    audio_wav = audio_wav[:, 0]

# Normalizar
audio_wav = audio_wav / np.max(np.abs(audio_wav))

print(f'Frecuencia de muestreo: {fs_wav} Hz')
print(f'Duración: {len(audio_wav)/fs_wav:.2f} s')
print(f'Número de muestras: {len(audio_wav)}')

In [ ]:
# Graficar señal de audio WAV
t_wav = np.arange(len(audio_wav)) / fs_wav

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_wav, y=audio_wav, mode='lines'))
fig.update_layout(
    title='Señal de audio - Owl.wav',
    xaxis_title='Tiempo (s)',
    yaxis_title='Amplitud',
    template='plotly_white'
)
fig.show()

In [ ]:
# Periodograma del archivo WAV
f_wav, Pxx_wav = signal.periodogram(audio_wav, fs_wav, window='hann', scaling='density')

fig = go.Figure()
fig.add_trace(go.Scatter(x=f_wav, y=Pxx_wav, mode='lines'))
fig.update_layout(
    title='Periodograma - Owl.wav',
    xaxis_title='Frecuencia (Hz)',
    yaxis_title='PSD (V²/Hz)',
    xaxis_range=[0, fs_wav/2],
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

In [ ]:
# Espectrograma del archivo WAV
f_spec, t_spec, Sxx = signal.spectrogram(audio_wav, fs_wav, window='hann', 
                                          nperseg=1024, noverlap=512)

fig = go.Figure(data=go.Heatmap(
    z=10*np.log10(Sxx),
    x=t_spec,
    y=f_spec,
    colorscale='Viridis',
    colorbar=dict(title='Potencia (dB)')
))

fig.update_layout(
    title='Espectrograma - Owl.wav',
    xaxis_title='Tiempo (s)',
    yaxis_title='Frecuencia (Hz)',
    yaxis_range=[0, fs_wav/2],
    template='plotly_white',
    height=500
)
fig.show()

In [ ]:
# Espectrograma con diferentes configuraciones de ventana
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        'Espectrograma - Ventana corta (nperseg=256)',
        'Espectrograma - Ventana media (nperseg=1024)',
        'Espectrograma - Ventana larga (nperseg=4096)'
    ),
    vertical_spacing=0.08,
    specs=[[{'type': 'heatmap'}], [{'type': 'heatmap'}], [{'type': 'heatmap'}]]
)

# Ventana corta
f1, t1, Sxx1 = signal.spectrogram(audio_wav, fs_wav, window='hann', 
                                    nperseg=256, noverlap=128)
fig.add_trace(go.Heatmap(
    z=10*np.log10(Sxx1),
    x=t1,
    y=f1,
    colorscale='Viridis',
    showscale=True,
    colorbar=dict(title='Potencia (dB)', y=0.83, len=0.25)
), row=1, col=1)

# Ventana media
f2, t2, Sxx2 = signal.spectrogram(audio_wav, fs_wav, window='hann', 
                                    nperseg=1024, noverlap=512)
fig.add_trace(go.Heatmap(
    z=10*np.log10(Sxx2),
    x=t2,
    y=f2,
    colorscale='Viridis',
    showscale=True,
    colorbar=dict(title='Potencia (dB)', y=0.5, len=0.25)
), row=2, col=1)

# Ventana larga
f3, t3, Sxx3 = signal.spectrogram(audio_wav, fs_wav, window='hann', 
                                    nperseg=4096, noverlap=2048)
fig.add_trace(go.Heatmap(
    z=10*np.log10(Sxx3),
    x=t3,
    y=f3,
    colorscale='Viridis',
    showscale=True,
    colorbar=dict(title='Potencia (dB)', y=0.17, len=0.25)
), row=3, col=1)

fig.update_xaxes(title_text='Tiempo (s)', row=3, col=1)
fig.update_yaxes(title_text='Frecuencia (Hz)', range=[0, fs_wav/2], row=1, col=1)
fig.update_yaxes(title_text='Frecuencia (Hz)', range=[0, fs_wav/2], row=2, col=1)
fig.update_yaxes(title_text='Frecuencia (Hz)', range=[0, fs_wav/2], row=3, col=1)

fig.update_layout(
    height=1200,
    template='plotly_white'
)
fig.show()

In [ ]:
# Cargar archivo MP3
audio_mp3, fs_mp3 = librosa.load('../../data/cuadrado.mp3', sr=None)

print(f'Frecuencia de muestreo: {fs_mp3} Hz')
print(f'Duración: {len(audio_mp3)/fs_mp3:.2f} s')
print(f'Número de muestras: {len(audio_mp3)}')

In [ ]:
# Graficar señal de audio MP3
t_mp3 = np.arange(len(audio_mp3)) / fs_mp3

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_mp3, y=audio_mp3, mode='lines'))
fig.update_layout(
    title='Señal de audio - cuadrado.mp3',
    xaxis_title='Tiempo (s)',
    yaxis_title='Amplitud',
    template='plotly_white'
)
fig.show()

In [ ]:
# Periodograma del archivo MP3
f_mp3, Pxx_mp3 = signal.periodogram(audio_mp3, fs_mp3, window='hann', scaling='density')

fig = go.Figure()
fig.add_trace(go.Scatter(x=f_mp3, y=Pxx_mp3, mode='lines'))
fig.update_layout(
    title='Periodograma - cuadrado.mp3',
    xaxis_title='Frecuencia (Hz)',
    yaxis_title='PSD (V²/Hz)',
    xaxis_range=[0, fs_mp3/2],
    yaxis_type='log',
    template='plotly_white'
)
fig.show()